In [ ]:
{
 "nbformat": 4,
 "nbformat_minor": 5,
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "cells": [
  {
   "cell_type": "markdown",
   "id": "cell-01",
   "metadata": {},
   "source": [
    "# baza500 — FAQ Scraper: 5 polskich sklepów meblowych\n",
    "\n",
    "**Cel:** Zebranie 500 unikalnych par pytanie/odpowiedź z kart FAQ 5 (docelowo 20) sklepów meblowych w Polsce.\n",
    "\n",
    "## Zbadane sklepy i struktury HTML\n",
    "\n",
    "| Sklep | URL FAQ | Pytanie (selektor) | Odpowiedź (selektor) |\n",
    "|---|---|---|---|\n",
    "| **Mebligo** | `/content/9-najczesciej-zadawane-pytania-faq` | `div[id^=\"ac-\"] button[id^=\"ac-trigger-\"]` | następne rodzeństwo `div` w bloku akordeonu |\n",
    "| **Stolar Meble** | `/faq/` | `button` w `div#content` | następny element po `button` |\n",
    "| **MeblujemyDOM** | `/pl/help/faq-najczesciej-zadawane-pytania-2` | elementy z `?` w `div#content` (nie-linki) | kolejne bloki tekstowe do następnego `?` |\n",
    "| **SalonMeblowy.net** | `/faq.ehtml` | `h2`, `h3` w `div#afaq` | kolejne `p`/`div` do następnego nagłówka |\n",
    "| **MebleM4** | `/faq-najczesciej-zadawane-pytania-w-meblem4-pl,p39.html` | elementy pasujące do `^\\d+\\.\\s+` | kolejne bloki tekstowe do następnego numeru |\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "cell-02",
   "metadata": {},
   "source": [
    "## Instalacja zależności"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-03",
   "metadata": {},
   "outputs": [],
   "source": [
    "!pip install requests beautifulsoup4 pandas rapidfuzz"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "cell-04",
   "metadata": {},
   "source": [
    "## Importy i konfiguracja"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-05",
   "metadata": {},
   "outputs": [],
   "source": [
    "import re\n",
    "import requests\n",
    "from bs4 import BeautifulSoup\n",
    "import pandas as pd\n",
    "\n",
    "HEADERS = {\n",
    "    \"User-Agent\": (\n",
    "        \"Mozilla/5.0 (Windows NT 10.0; Win64; x64) \"\n",
    "        \"AppleWebKit/537.36 (KHTML, like Gecko) \"\n",
    "        \"Chrome/124.0.0.0 Safari/537.36\"\n",
    "    )\n",
    "}\n",
    "\n",
    "def get_soup(url: str) -> BeautifulSoup:\n",
    "    r = requests.get(url, headers=HEADERS, timeout=20)\n",
    "    r.raise_for_status()\n",
    "    return BeautifulSoup(r.text, \"html.parser\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "cell-06",
   "metadata": {},
   "source": [
    "## 1. Mebligo.pl\n",
    "\n",
    "**Struktura:** akordeon `div[id^=\"ac-N\"]` → `button[id^=\"ac-trigger-N\"]` (pytanie) + ukryty `div` z odpowiedzią (obecny w HTML mimo CSS display:none)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-07",
   "metadata": {},
   "outputs": [],
   "source": [
    "def parse_mebligo() -> list:\n",
    "    url = \"https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq\"\n",
    "    soup = get_soup(url)\n",
    "    qas = []\n",
    "\n",
    "    for block in soup.find_all(\"div\", id=re.compile(r\"^ac-\\d+$\")):\n",
    "        btn = block.find(\"button\", id=re.compile(r\"^ac-trigger-\"))\n",
    "        if not btn:\n",
    "            continue\n",
    "        question = btn.get_text(\" \", strip=True).rstrip(\" +\").strip()\n",
    "\n",
    "        answer_parts = []\n",
    "        for child in block.children:\n",
    "            if hasattr(child, \"name\") and child.name and child != btn.parent:\n",
    "                text = child.get_text(\" \", strip=True)\n",
    "                if text:\n",
    "                    answer_parts.append(text)\n",
    "        answer = \" \".join(answer_parts).strip()\n",
    "\n",
    "        if question and answer:\n",
    "            qas.append({\"shop\": \"Mebligo\", \"source_url\": url,\n",
    "                        \"question\": question, \"answer\": answer})\n",
    "    return qas\n",
    "\n",
    "r = parse_mebligo()\n",
    "print(f\"Mebligo: {len(r)} Q&A\")\n",
    "r[:2]"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "cell-08",
   "metadata": {},
   "source": [
    "## 2. StolarMeble.pl\n",
    "\n",
    "**Struktura:** `div#content` → `button` (pytanie) + następny element rodzeństwo (odpowiedź). Wszystkie elementy widoczne bezpośrednio w DOM."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-09",
   "metadata": {},
   "outputs": [],
   "source": [
    "def parse_stolar() -> list:\n",
    "    url = \"https://stolarmeble.pl/faq/\"\n",
    "    soup = get_soup(url)\n",
    "    qas = []\n",
    "\n",
    "    content = soup.find(\"div\", id=\"content\")\n",
    "    if not content:\n",
    "        return qas\n",
    "\n",
    "    for btn in content.find_all(\"button\"):\n",
    "        question = btn.get_text(\" \", strip=True)\n",
    "        if not question:\n",
    "            continue\n",
    "        answer_node = btn.find_next_sibling()\n",
    "        if not answer_node:\n",
    "            answer_node = btn.parent.find_next_sibling()\n",
    "        answer = answer_node.get_text(\" \", strip=True) if answer_node else \"\"\n",
    "\n",
    "        if question and answer:\n",
    "            qas.append({\"shop\": \"Stolar Meble\", \"source_url\": url,\n",
    "                        \"question\": question, \"answer\": answer})\n",
    "    return qas\n",
    "\n",
    "r = parse_stolar()\n",
    "print(f\"Stolar: {len(r)} Q&A\")\n",
    "r[:2]"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "cell-10",
   "metadata": {},
   "source": [
    "## 3. MeblujemyDOM.pl\n",
    "\n",
    "**Struktura:** `div#content` → naprzemienne bloki tekstowe: element z `?` = pytanie, kolejne bloki do następnego `?` = odpowiedź."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-11",
   "metadata": {},
   "outputs": [],
   "source": [
    "def parse_meblujemydom() -> list:\n",
    "    url = \"https://meblujemydom.pl/pl/help/faq-najczesciej-zadawane-pytania-2\"\n",
    "    soup = get_soup(url)\n",
    "    qas = []\n",
    "\n",
    "    content = soup.find(\"div\", id=\"content\")\n",
    "    if not content:\n",
    "        return qas\n",
    "\n",
    "    children = [c for c in content.children\n",
    "                if hasattr(c, \"name\") and c.name is not None]\n",
    "\n",
    "    i = 0\n",
    "    while i < len(children):\n",
    "        node = children[i]\n",
    "        text = node.get_text(\" \", strip=True)\n",
    "        is_toc_link = bool(node.find(\"a\")) and re.match(r\"^\\d+\\.\", text)\n",
    "        is_question = (\n",
    "            \"?\" in text\n",
    "            and not is_toc_link\n",
    "            and 10 < len(text) < 300\n",
    "            and not node.find(\"a\", href=True)\n",
    "        )\n",
    "        if is_question:\n",
    "            answer_parts = []\n",
    "            j = i + 1\n",
    "            while j < len(children):\n",
    "                next_text = children[j].get_text(\" \", strip=True)\n",
    "                if \"?\" in next_text and len(next_text) < 300:\n",
    "                    break\n",
    "                if next_text:\n",
    "                    answer_parts.append(next_text)\n",
    "                j += 1\n",
    "            answer = \" \".join(answer_parts).strip()\n",
    "            if text and answer:\n",
    "                qas.append({\"shop\": \"MeblujemyDOM\", \"source_url\": url,\n",
    "                            \"question\": text, \"answer\": answer})\n",
    "            i = j\n",
    "        else:\n",
    "            i += 1\n",
    "    return qas\n",
    "\n",
    "r = parse_meblujemydom()\n",
    "print(f\"MeblujemyDOM: {len(r)} Q&A\")\n",
    "r[:2]"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "cell-12",
   "metadata": {},
   "source": [
    "## 4. SalonMeblowy.net.pl\n",
    "\n",
    "**Struktura:** `div#afaq` → `h2`/`h3` = pytanie, kolejne `p`/`div`/`generic` do następnego nagłówka = odpowiedź."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-13",
   "metadata": {},
   "outputs": [],
   "source": [
    "def parse_salonmeblowy() -> list:\n",
    "    url = \"https://www.salonmeblowy.net.pl/faq.ehtml\"\n",
    "    soup = get_soup(url)\n",
    "    qas = []\n",
    "\n",
    "    afaq = soup.find(\"div\", id=\"afaq\")\n",
    "    if not afaq:\n",
    "        return qas\n",
    "\n",
    "    children = [c for c in afaq.children\n",
    "                if hasattr(c, \"name\") and c.name is not None]\n",
    "\n",
    "    i = 0\n",
    "    while i < len(children):\n",
    "        node = children[i]\n",
    "        if node.name in (\"h2\", \"h3\"):\n",
    "            question = node.get_text(\" \", strip=True)\n",
    "            answer_parts = []\n",
    "            j = i + 1\n",
    "            while j < len(children):\n",
    "                sibling = children[j]\n",
    "                if sibling.name in (\"h2\", \"h3\"):\n",
    "                    break\n",
    "                text = sibling.get_text(\" \", strip=True)\n",

In [9]:
!pip install jinja2


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Users\USER\anaconda3\python.exe -m pip install --upgrade pip


In [2]:
import pandas
print(pandas.__version__)

3.0.1


In [3]:
!pip install --upgrade pandas

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.8 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.8 MB 1.1 MB/s eta 0:00:09
   --- ------------------------------------ 0.8/9.8 MB 1.2 MB/s eta 0:00:08
   ---- ----------------------------------- 1.0/9.8 MB 1.2 MB/s eta 0:00:08
   ------ --------------------------------- 1.6/9.8 MB 1.4 MB/s eta 0:00:07
   ------- -------------------------------- 1.8/9.8 MB 1.4 MB/s eta 0:00:06
   -------- ------------------------------- 2.1/9.8 MB 1.4 MB/s eta 0:00:06
   ---------- ----------------------------- 2.6/9.8 MB 1.5 MB/s eta 0:00:05
   ----------- ---------------------------- 2.9/9.8 MB 1.5 MB/s eta 0:00:05
   ------------ --------------------------- 3.1/9.8 MB 1.5 MB/s eta 0:00:05
   ------------- -------------------------- 3.4/9.8 MB 1.5 MB/s eta 0:00:05
   --------------- --------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires pandas<3,>=1.4.0, but you have pandas 3.0.3 which is incompatible.

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Users\USER\anaconda3\python.exe -m pip install --upgrade pip


In [9]:
# === KOMPLEKSOWE BADANIE TABELI CSV (Wersja Bez Indeksu) ===

import pandas as pd

# 1. Wczytowanie danych
# Πrawidłowa ścieżka z Twojego przykładu
df = pd.read_csv(r"C:\1\T4\jdszr24-grupa-4\SKRAPOWANIE\SKRAP_DED\DATA\faq_output_1.csv")

#選項Display options: Ukrywanie tekstu, aby nie był uciet
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 120)

# Helper: Funkcja do wyświetlania DF bez indeksu (0, 1, 2...)
def show(df):
    return df.style.hide(axis="index")

# 2. Podstawowe informacje
print("=" * 60)
print("1. PODSTAWOWE INFORMACJE")
print("=" * 60)
print(f"Liczba wierszy: {df.shape[0]}")
print(f"Liczba kolumn: {df.shape[1]}")
print(f"Całkowita liczba obserwacji: {df.shape[0] * df.shape[1]}")

# 3. Nazwy kolumn
print("=" * 60)
print("2. NAZWY KOLUMN")
print("=" * 60)
print(df.columns.tolist())

# 4. Podgląd danych (pierwsze 5 wierszy) - BEZ INDEKSU
print("=" * 60)
print("3. PODGLĄD DANYCH (pierwsze 5 wierszy)")
print("=" * 60)
display(show(df.head()))

# 4. Podgląd danych (ostatnie 5 wierszy) - BEZ INDEKSU
print("=" * 60)
print("4. PODGLĄD DANYCH (ostatnie 5 wierszy)")
print("=" * 60)
display(show(df.tail()))

# 5. Informacje o typach danych i brakujących wartościach - BEZ INDEKSU
print("=" * 60)
print("5. TYPOW DANYCH I BRAKUJĄCE WARTOŚCI")
print("=" * 60)
info_df = pd.DataFrame({
    'Typ danych': df.dtypes,
    'Liczba brakujących': df.isnull().sum(),
    '% brakujących': (df.isnull().sum() / len(df) * 100).round(2),
    'Unikalne wartości': df.nunique()
})
display(show(info_df))

# 6. Statystyki opisowe (dla kolumn numerycznych) - BEZ INDEKSU
print("=" * 60)
print("6. STATYSTYKI OPISOWE (kolumny numeryczne)")
print("=" * 60)
display(show(df.describe()))

# 7. Statystyki dla kolumn tekstowych - BEZ INDEKSU
print("=" * 60)
print("7. STATYSTYKI (kolumny tekstowe)")
print("=" * 60)
display(show(df.describe(include='object')))

# 8. Zakres danych
print("=" * 60)
print("8. ZAKRES DANYCH")
print("=" * 60)
for col in df.columns:
    if df[col].dtype in ['int64', 'float64']:
        print(f"{col}: {df[col].min()} -> {df[col].max()}")
    elif df[col].dtype == 'object':
        print(f"{col}: {df[col].nunique()} unikalnych wartości")

# 9. Sprawdź potencjalne kolumny daty
print("=" * 60)
print("9. POTENCJALNE KOLUMNY DATY")
print("=" * 60)
date_cols = df.columns[df.columns.str.contains('date|time|month|year|Date|Time|Month|Year', case=False)]
print(f"Potencjalne kolumny daty: {date_cols.tolist()}")

# 10. Jeśli to szereg czasowy - dodatkowe informacje
print("=" * 60)
print("10. SZCZEGOLNE INFORMACJE (szereg czasowy)")
print("=" * 60)
if len(date_cols) > 0:
    date_col = date_cols[0]
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df_ts = df.set_index(date_col).sort_index() # Użyłem df_ts żeby nie zmodyfikować głównego df
    print(f"Indeks czasowy: {df_ts.index.min()} -> {df_ts.index.max()}")
    print(f"Liczba obserwacji: {len(df_ts)}")
    print(f"Częstotliwość: {df_ts.index.freq}")

# 11. Korelacje (jeśli są kolumny numeryczne)
if df.select_dtypes(include=['int64', 'float64']).shape[1] > 1:
    print("=" * 60)
    print("11. KORELACJE MIĘDZY KOLUMNAMI NUMERYCZNYMI")
    print("=" * 60)
    corr_df = df.select_dtypes(include=['int64', 'float64']).corr()
    display(show(corr_df))

print("=" * 60)
print("=== BADANIE ZAKOŃCZONE ===")
print("=" * 60)

# === TRZY NOWE SPOSÓBY WYKRYŚLENIA (TRYB HEAD) ===

# Wersja 1: Losowe kolumny (początek + koniec) - BEZ INDEKSU
cols = df.columns[:3].tolist() + df.columns[-3:].tolist()
print("=" * 60)
print("NOWE 1:แนะUd clé (początek + koniec sloužców)")
print("=" * 60)
display(show(df[cols].head()))

# Wersja 2: Transpozycja (kolumny jako wiersze) - BEZ INDEKSU
print("=" * 60)
print("NOWE 2: Transpozycja ( detalący każdej kolumny)")
print("=" * 60)
display(show(df.head().T))

# Wersja 3: UTF-rows z brakami (debugowanie) - BEZ INDEKSU
print("=" * 60)
print("NOWE 3: Wiersz z brakującymi wartościami")
print("=" * 60)
null_rows = df[df.isnull().any(axis=1)]
if null_rows.empty:
    print("Nie ma brakujących wartości w tym zbiorze.")
else:
    display(show(null_rows.head()))

1. PODSTAWOWE INFORMACJE
Liczba wierszy: 3
Liczba kolumn: 4
Całkowita liczba obserwacji: 12
2. NAZWY KOLUMN
['shop', 'source_url', 'question', 'answer']
3. PODGLĄD DANYCH (pierwsze 5 wierszy)


shop,source_url,question,answer
shop,source_url,question,answer
Mebligo,https://mebligo.pl/faq1,Jakie są czasy dostawy?,Dostawa w 2-3 dni robocze.
Mebligo,https://mebligo.pl/faq2,Czy mogę zwrócić produkt?,"Tak, w ciągu 14 dni bez wyjaśnienia przyczyny."


4. PODGLĄD DANYCH (ostatnie 5 wierszy)


shop,source_url,question,answer
shop,source_url,question,answer
Mebligo,https://mebligo.pl/faq1,Jakie są czasy dostawy?,Dostawa w 2-3 dni robocze.
Mebligo,https://mebligo.pl/faq2,Czy mogę zwrócić produkt?,"Tak, w ciągu 14 dni bez wyjaśnienia przyczyny."


5. TYPOW DANYCH I BRAKUJĄCE WARTOŚCI


...

8. ZAKRES DANYCH
9. POTENCJALNE KOLUMNY DATY
Potencjalne kolumny daty: []
10. SZCZEGOLNE INFORMACJE (szereg czasowy)
=== BADANIE ZAKOŃCZONE ===
NOWE 1:แนะUd clé (początek + koniec sloužców)


shop,source_url,question,source_url,question,answer
shop,source_url,question,source_url,question,answer
Mebligo,https://mebligo.pl/faq1,Jakie są czasy dostawy?,https://mebligo.pl/faq1,Jakie są czasy dostawy?,Dostawa w 2-3 dni robocze.
Mebligo,https://mebligo.pl/faq2,Czy mogę zwrócić produkt?,https://mebligo.pl/faq2,Czy mogę zwrócić produkt?,"Tak, w ciągu 14 dni bez wyjaśnienia przyczyny."


NOWE 2: Transpozycja ( detalący każdej kolumny)


0,1,2
shop,Mebligo,Mebligo
source_url,https://mebligo.pl/faq1,https://mebligo.pl/faq2
question,Jakie są czasy dostawy?,Czy mogę zwrócić produkt?
answer,Dostawa w 2-3 dni robocze.,"Tak, w ciągu 14 dni bez wyjaśnienia przyczyny."


NOWE 3: Wiersz z brakującymi wartościami
Nie ma brakujących wartości w tym zbiorze.


In [10]:
import jinja2
print(jinja2.__version__)

3.1.6


In [8]:
import sys
print(sys.executable)


c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\python.exe


In [11]:
# === KOMPLEKSOWE BADANIE FAQ / CZAT CSV Z CZYSZCZENIEM TEKSTU ===

import pandas as pd
import re

# 1. Wczytanie danych z obsługą polskich znaków
df = pd.read_csv(
    r"C:\1\T4\jdszr24-grupa-4\SKRAPOWANIE\SKRAP_DED\DATA\faq_output_1.csv",
    encoding="utf-8"  # jeśli zobaczysz krzaki, zmień na: encoding="cp1250"
)

# 2. Ustawienia wyświetlania (dla czytelności długich tekstów)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 140)

# 3. Konfiguracja kolumn QA (dopasuj jeśli masz inne nazwy)
QA_COLS = {
    "shop": "shop",
    "url": "source_url",
    "q": "question",
    "a": "answer",
}

# 4. Funkcja czyszcząca tekst
def clean_text(s: str) -> str:
    if pd.isna(s):
        return s

    # rzutowanie na string
    s = str(s)

    # usunięcie znaczników HTML typu <br>, <p>, <span> itd.
    s = re.sub(r"<[^>]+>", " ", s)

    # dekodowanie typowych encji HTML (proste przypadki; pełne → html.unescape)
    html_entities = {
        "&nbsp;": " ",
        "&amp;": "&",
        "&quot;": '"',
        "&apos;": "'",
        "&lt;": "<",
        "&gt;": ">",
    }
    for ent, rep in html_entities.items():
        s = s.replace(ent, rep)

    # usunięcie twardych spacji i dziwnych whitespace (np. \xa0, \r)
    s = s.replace("\xa0", " ")
    s = s.replace("\r", " ")

    # zamiana wielu białych znaków (spacje, taby, \n) na jedną spację
    s = re.sub(r"\s+", " ", s)

    # usunięcie niewidocznych znaków sterujących (poza podstawowym whitespace)
    s = "".join(ch for ch in s if ch == " " or not ch.isspace() or ch in "\t\n")

    # przycięcie przodka i końca
    s = s.strip()

    return s

# 5. Zastosowanie czyszczenia do pytań i odpowiedzi
for col in [QA_COLS["q"], QA_COLS["a"]]:
    df[col] = df[col].apply(clean_text)

# 6. Dodatkowe uproszczenie: usunięcie pustych stringów → NaN (łatwiej filtrować)
for col in [QA_COLS["q"], QA_COLS["a"]]:
    df.loc[df[col].astype(str).str.strip() == "", col] = pd.NA

# 7. Podstawowe informacje o zbiorze QA
print("=" * 60)
print("A1. PODSTAWOWE INFO O PYTANIACH I ODPOWIEDZIACH")
print("=" * 60)

n_rows = len(df)
n_shops = df[QA_COLS["shop"]].nunique()
n_urls  = df[QA_COLS["url"]].nunique()
n_q     = df[QA_COLS["q"]].nunique()
n_a     = df[QA_COLS["a"]].nunique()

print(f"Liczba rekordów (wierszy): {n_rows}")
print(f"Liczba unikalnych sklepów: {n_shops}")
print(f"Liczba unikalnych URL-i:   {n_urls}")
print(f"Liczba unikalnych pytań:   {n_q}")
print(f"Liczba unikalnych odpowiedzi: {n_a}")

print("\nPróbka pytań i odpowiedzi po czyszczeniu:")
display(
    df[[QA_COLS["shop"], QA_COLS["q"], QA_COLS["a"]]]
    .head(10)
    .reset_index(drop=True)
)

# 8. Duplikaty pytań i par Q&A
print("=" * 60)
print("A2. DUPLIKATY PYTAŃ I PAR Q&A")
print("=" * 60)

dup_q = df.duplicated(subset=[QA_COLS["q"]]).sum()
dup_qa = df.duplicated(subset=[QA_COLS["q"], QA_COLS["a"]]).sum()

print(f"Liczba zduplikowanych PYTAŃ (po samym 'question'): {dup_q}")
print(f"Liczba zduplikowanych PAR Q&A:                    {dup_qa}")

q_counts = df[QA_COLS["q"]].value_counts()
print("\nNajczęściej powtarzające się pytania:")
print(q_counts[q_counts > 1].head(10))

# 9. Długości pytań i odpowiedzi
print("=" * 60)
print("A3. DŁUGOŚCI PYTAŃ I ODPOWIEDZI (znaki)")
print("=" * 60)

df["question_len"] = df[QA_COLS["q"]].astype(str).str.len()
df["answer_len"]   = df[QA_COLS["a"]].astype(str).str.len()

print("Statystyki długości PYTAŃ:")
print(df["question_len"].describe())

print("\nStatystyki długości ODPOWIEDZI:")
print(df["answer_len"].describe())

print("\nNajkrótsze PYTANIA (top 5) po czyszczeniu:")
display(
    df[[QA_COLS["shop"], QA_COLS["q"], "question_len"]]
    .sort_values("question_len")
    .head(5)
    .reset_index(drop=True)
)

print("\nNajdłuższe PYTANIA (top 5) po czyszczeniu:")
display(
    df[[QA_COLS["shop"], QA_COLS["q"], "question_len"]]
    .sort_values("question_len", ascending=False)
    .head(5)
    .reset_index(drop=True)
)

print("\nNajkrótsze ODPOWIEDZI (top 5) po czyszczeniu:")
display(
    df[[QA_COLS["shop"], QA_COLS["a"], "answer_len"]]
    .sort_values("answer_len")
    .head(5)
    .reset_index(drop=True)
)

print("\nNajdłuższe ODPOWIEDZI (top 5) po czyszczeniu:")
display(
    df[[QA_COLS["shop"], QA_COLS["a"], "answer_len"]]
    .sort_values("answer_len", ascending=False)
    .head(5)
    .reset_index(drop=True)
)

# 10. Braki w treści po czyszczeniu
print("=" * 60)
print("A4. BRAKI W PYTANIACH I ODPOWIEDZIACH (po czyszczeniu)")
print("=" * 60)

missing_q = df[QA_COLS["q"]].isna().sum()
missing_a = df[QA_COLS["a"]].isna().sum()

print(f"Brakujące (NaN) PYTANIA:    {missing_q}")
print(f"Brakujące (NaN) ODPOWIEDZI: {missing_a}")

print("\nPrzykładowe rekordy z brakami:")
bad_rows = df[df[QA_COLS["q"]].isna() | df[QA_COLS["a"]].isna()]
display(bad_rows.head(10).reset_index(drop=True))

# 11. Rozkład FAQ po sklepie i URL
print("=" * 60)
print("A5. LICZBA PYTAŃ NA SKLEP I URL")
print("=" * 60)

print("Top 10 sklepów po liczbie pytań:")
display(
    df.groupby(QA_COLS["shop"])[QA_COLS["q"]]
    .count()
    .sort_values(ascending=False)
    .head(10)
    .to_frame("liczba_pytan")
    .reset_index()
)

print("\nTop 10 URL-i po liczbie pytań:")
display(
    df.groupby(QA_COLS["url"])[QA_COLS["q"]]
    .count()
    .sort_values(ascending=False)
    .head(10)
    .to_frame("liczba_pytan")
    .reset_index()
)

# 12. Pierwsze słowo pytania (proxy tematu)
print("=" * 60)
print("A6. PIERWSZE SŁOWO PYTANIA")
print("=" * 60)

df["q_first_word"] = (
    df[QA_COLS["q"]]
    .astype(str)
    .str.strip()
    .str.split()
    .str[0]
    .str.lower()
)

fw_counts = df["q_first_word"].value_counts().head(20)
print("Najczęstsze pierwsze słowa w pytaniach:")
print(fw_counts)

print("\nPrzykładowe pytania dla najpopularniejszego pierwszego słowa:")
if not fw_counts.empty:
    top_fw = fw_counts.index[0]
    display(
        df[df["q_first_word"] == top_fw][[QA_COLS["q"], QA_COLS["a"]]]
        .head(10)
        .reset_index(drop=True)
)

# 13. Losowa próbka Q&A (po czyszczeniu)
print("=" * 60)
print("A7. LOSOWA PRÓBKA PYTAŃ I ODPOWIEDZI (po czyszczeniu)")
print("=" * 60)

sample_qa = df[[QA_COLS["shop"], QA_COLS["q"], QA_COLS["a"]]].sample(
    n=min(10, len(df)),
    random_state=42
)
display(sample_qa.reset_index(drop=True))

print("=" * 60)
print("=== BADANIE FAQ + CZYSZCZENIE ZAKOŃCZONE ===")
print("=" * 60)

A1. PODSTAWOWE INFO O PYTANIACH I ODPOWIEDZIACH
Liczba rekordów (wierszy): 3
Liczba unikalnych sklepów: 2
Liczba unikalnych URL-i:   3
Liczba unikalnych pytań:   3
Liczba unikalnych odpowiedzi: 3

Próbka pytań i odpowiedzi po czyszczeniu:


,shop,question,answer
0,shop,question,answer
1,Mebligo,Jakie są czasy dostawy?,Dostawa w 2-3 dni robocze.
2,Mebligo,Czy mogę zwrócić produkt?,"Tak, w ciągu 14 dni bez wyjaśnienia przyczyny."


A2. DUPLIKATY PYTAŃ I PAR Q&A
Liczba zduplikowanych PYTAŃ (po samym 'question'): 0
Liczba zduplikowanych PAR Q&A:                    0

Najczęściej powtarzające się pytania:
Series([], Name: count, dtype: int64)
A3. DŁUGOŚCI PYTAŃ I ODPOWIEDZI (znaki)
Statystyki długości PYTAŃ:
count     3.000000
mean     18.666667
std       9.291573
min       8.000000
25%      15.500000
50%      23.000000
75%      24.000000
max      25.000000
Name: question_len, dtype: float64

Statystyki długości ODPOWIEDZI:
count     3.0
mean     26.0
std      20.0
min       6.0
25%      16.0
50%      26.0
75%      36.0
max      46.0
Name: answer_len, dtype: float64

Najkrótsze PYTANIA (top 5) po czyszczeniu:


,shop,question,question_len
0,shop,question,8
1,Mebligo,Jakie są czasy dostawy?,23
2,Mebligo,Czy mogę zwrócić produkt?,25



Najdłuższe PYTANIA (top 5) po czyszczeniu:


...

A6. PIERWSZE SŁOWO PYTANIA
Najczęstsze pierwsze słowa w pytaniach:
q_first_word
question    1
jakie       1
czy         1
Name: count, dtype: int64

Przykładowe pytania dla najpopularniejszego pierwszego słowa:


,question,answer
0,question,answer


A7. LOSOWA PRÓBKA PYTAŃ I ODPOWIEDZI (po czyszczeniu)


,shop,question,answer
0,shop,question,answer
1,Mebligo,Jakie są czasy dostawy?,Dostawa w 2-3 dni robocze.
2,Mebligo,Czy mogę zwrócić produkt?,"Tak, w ciągu 14 dni bez wyjaśnienia przyczyny."


=== BADANIE FAQ + CZYSZCZENIE ZAKOŃCZONE ===


In [18]:
# === KOMPLEKSOWE, ELASTYCZNE BADANIE FAQ / CZAT CSV ===

import pandas as pd
import re

# 1. Wczytanie danych (obsługa polskich znaków, autodetekcja separatora)
csv_path = r"C:\1\T4\jdszr24-grupa-4\SKRAPOWANIE\SKRAP_DED\DATA\faq_output.csv"

df = pd.read_csv(
    csv_path,
    encoding="utf-8",   # jeśli będą krzaki, zmień na "cp1250"
    engine="python",
    sep=None,           # autodetekcja separatora
)

# 2. Ustawienia wyświetlania (czytelność)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 140)

print("KOLUMNY W DF:", list(df.columns))

# 3. Automatyczna konfiguracja kolumn FAQ / czatu
#    (działa niezależnie od liczby kolumn i odmian nazw)

cols_lower = {c.lower().strip(): c for c in df.columns}

def find_col(candidates):
    """Zwróć fizyczną nazwę kolumny pasującą do którejś z podanych etykiet logicznych."""
    for cand in candidates:
        if cand in cols_lower:
            return cols_lower[cand]
    return None

QA_COLS = {
    "shop": find_col(["shop", "sklep"]),
    "url":  find_col(["source_url", "url", "link", "href"]),
    "q":    find_col(["question", "pytanie", "query", "user_message"]),
    "a":    find_col(["answer", "odpowiedz", "odpowiedź", "response", "bot_message"]),
}

print("ZMAPOWANE KOLUMNY QA_COLS:", QA_COLS)

# 4. Funkcja czyszcząca tekst (HTML, encje, whitespace)
def clean_text(s: str) -> str:
    if pd.isna(s):
        return s
    s = str(s)

    # usuń proste znaczniki HTML
    s = re.sub(r"<[^>]+>", " ", s)

    # podstawowe encje HTML
    html_entities = {
        "&nbsp;": " ",
        "&amp;": "&",
        "&quot;": '"',
        "&apos;": "'",
        "&lt;": "<",
        "&gt;": ">",
    }
    for ent, rep in html_entities.items():
        s = s.replace(ent, rep)

    # twarde spacje, CR
    s = s.replace("\xa0", " ")
    s = s.replace("\r", " ")

    # wielokrotne białe znaki → jedna spacja
    s = re.sub(r"\s+", " ", s)

    # przycięcie
    return s.strip()

# 5. Czyszczenie tekstu we wszystkich kolumnach typu tekstowego (pełna elastyczność)

for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].apply(clean_text)
    df.loc[df[col].astype(str).str.strip() == "", col] = pd.NA

# 6. Podstawowe informacje (na całym df)
print("=" * 60)
print("1. PODSTAWOWE INFORMACJE O TABELI (WSZYSTKIE REKORDY)")
print("=" * 60)
print(f"Liczba wierszy: {df.shape[0]}")
print(f"Liczba kolumn: {df.shape[1]}")
print(f"Całkowita liczba obserwacji: {df.shape[0] * df.shape[1]}")

# 7. Nazwy kolumn
print("=" * 60)
print("2. NAZWY KOLUMN")
print("=" * 60)
print(df.columns.tolist())

# 8. Podgląd danych – 3 pierwsze i 3 ostatnie wiersze
print("=" * 60)
print("3. PODGLĄD DANYCH (pierwsze 3 wiersze)")
print("=" * 60)
display(df.head(3).reset_index(drop=True))

print("=" * 60)
print("4. PODGLĄD DANYCH (ostatnie 3 wiersze)")
print("=" * 60)
display(df.tail(3).reset_index(drop=True))

# 9. Typy danych i braki – na całym df
print("=" * 60)
print("5. TYPY DANYCH I BRAKUJĄCE WARTOŚCI (WSZYSTKIE REKORDY)")
print("=" * 60)
info_df = pd.DataFrame({
    "Typ danych": df.dtypes,
    "Liczba brakujących": df.isnull().sum(),
    "% brakujących": (df.isnull().sum() / len(df) * 100).round(2),
    "Unikalne wartości": df.nunique()
})
display(info_df.reset_index().rename(columns={"index": "Kolumna"}))

# 10. Statystyki numeryczne – całe df (jeśli są)
print("=" * 60)
print("6. STATYSTYKI OPISOWE (kolumny numeryczne, WSZYSTKIE REKORDY)")
print("=" * 60)
num = df.select_dtypes(include=["int64", "float64"])
if not num.empty:
    display(num.describe())
else:
    print("Brak kolumn numerycznych.")

# 11. Statystyki tekstowe – całe df
print("=" * 60)
print("7. STATYSTYKI TEKSTOWE (WSZYSTKIE REKORDY)")
print("=" * 60)
display(df.describe(include="object"))

# 12. Zakres danych – całe df
print("=" * 60)
print("8. ZAKRES DANYCH (WSZYSTKIE REKORDY)")
print("=" * 60)
for col in df.columns:
    if df[col].dtype in ["int64", "float64"]:
        print(f"{col}: {df[col].min()} -> {df[col].max()}")
    elif df[col].dtype == "object":
        print(f"{col}: {df[col].nunique()} unikalnych wartości")

# 13. Specjalne EDA dla FAQ / czatu – tylko jeśli mamy Q/A

has_q = QA_COLS["q"] is not None
has_a = QA_COLS["a"] is not None
has_shop = QA_COLS["shop"] is not None
has_url = QA_COLS["url"] is not None

if has_q and has_a:
    print("=" * 60)
    print("A1. INFO O PYTANIACH I ODPOWIEDZIACH (WSZYSTKIE REKORDY)")
    print("=" * 60)

    n_rows = len(df)
    n_shops = df[QA_COLS["shop"]].nunique() if has_shop else None
    n_urls  = df[QA_COLS["url"]].nunique() if has_url else None
    n_q     = df[QA_COLS["q"]].nunique()
    n_a     = df[QA_COLS["a"]].nunique()

    print(f"Liczba rekordów:              {n_rows}")
    if has_shop:
        print(f"Liczba unikalnych sklepów:    {n_shops}")
    if has_url:
        print(f"Liczba unikalnych URL-i:      {n_urls}")
    print(f"Liczba unikalnych PYTAŃ:      {n_q}")
    print(f"Liczba unikalnych ODPOWIEDZI: {n_a}")

    # duplikaty – globalnie
    print("=" * 60)
    print("A2. DUPLIKATY PYTAŃ I PAR Q&A (WSZYSTKIE REKORDY)")
    print("=" * 60)

    dup_q  = df.duplicated(subset=[QA_COLS["q"]]).sum()
    dup_qa = df.duplicated(subset=[QA_COLS["q"], QA_COLS["a"]]).sum()

    print(f"Liczba zduplikowanych PYTAŃ:   {dup_q}")
    print(f"Liczba zduplikowanych PAR Q&A: {dup_qa}")

    # długości – globalnie
    print("=" * 60)
    print("A3. DŁUGOŚCI PYTAŃ I ODPOWIEDZI (WSZYSTKIE REKORDY)")
    print("=" * 60)

    df["question_len"] = df[QA_COLS["q"]].astype(str).str.len()
    df["answer_len"]   = df[QA_COLS["a"]].astype(str).str.len()

    print("Statystyki długości PYTAŃ:")
    print(df["question_len"].describe())

    print("\nStatystyki długości ODPOWIEDZI:")
    print(df["answer_len"].describe())

    print("\nNajkrótsze PYTANIA (top 3):")
    display(
        df[[QA_COLS["q"], "question_len"]]
        .sort_values("question_len")
        .head(3)
        .reset_index(drop=True)
    )

    print("\nNajdłuższe PYTANIA (top 3):")
    display(
        df[[QA_COLS["q"], "question_len"]]
        .sort_values("question_len", ascending=False)
        .head(3)
        .reset_index(drop=True)
    )

    print("\nNajkrótsze ODPOWIEDZI (top 3):")
    display(
        df[[QA_COLS["a"], "answer_len"]]
        .sort_values("answer_len")
        .head(3)
        .reset_index(drop=True)
    )

    print("\nNajdłuższe ODPOWIEDZI (top 3):")
    display(
        df[[QA_COLS["a"], "answer_len"]]
        .sort_values("answer_len", ascending=False)
        .head(3)
        .reset_index(drop=True)
    )

    # braki – globalnie
    print("=" * 60)
    print("A4. BRAKI W PYTANIACH I ODPOWIEDZIACH (WSZYSTKIE REKORDY)")
    print("=" * 60)

    missing_q = df[QA_COLS["q"]].isna().sum()
    missing_a = df[QA_COLS["a"]].isna().sum()

    print(f"Brakujące (NaN) PYTANIA:    {missing_q}")
    print(f"Brakujące (NaN) ODPOWIEDZI: {missing_a}")

    bad_rows = df[df[QA_COLS["q"]].isna() | df[QA_COLS["a"]].isna()]
    print("\nPrzykładowe rekordy z brakami (max 3):")
    display(bad_rows.head(3).reset_index(drop=True))

    # rozkład po sklepie/URL – jeśli dostępne
    if has_shop:
        print("=" * 60)
        print("A5. LICZBA PYTAŃ NA SKLEP (WSZYSTKIE REKORDY)")
        print("=" * 60)
        display(
            df.groupby(QA_COLS["shop"])[QA_COLS["q"]]
            .count()
            .sort_values(ascending=False)
            .head(10)
            .to_frame("liczba_pytan")
            .reset_index()
        )

    if has_url:
        print("\nTop 10 URL-i po liczbie pytań:")
        display(
            df.groupby(QA_COLS["url"])[QA_COLS["q"]]
            .count()
            .sort_values(ascending=False)
            .head(10)
            .to_frame("liczba_pytan")
            .reset_index()
        )

    # pierwsze słowo pytania – globalnie
    print("=" * 60)
    print("A6. PIERWSZE SŁOWO PYTANIA (WSZYSTKIE REKORDY)")
    print("=" * 60)

    df["q_first_word"] = (
        df[QA_COLS["q"]]
        .astype(str)
        .str.strip()
        .str.split()
        .str[0]
        .str.lower()
    )

    fw_counts = df["q_first_word"].value_counts().head(20)
    print("Najczęstsze pierwsze słowa w pytaniach:")
    print(fw_counts)

    print("\nPrzykładowe pytania dla najpopularniejszego pierwszego słowa (max 3):")
    if not fw_counts.empty:
        top_fw = fw_counts.index[0]
        display(
            df[df["q_first_word"] == top_fw][[QA_COLS["q"], QA_COLS["a"]]]
            .head(3)
            .reset_index(drop=True)
        )

else:
    print("=" * 60)
    print("A*. BRAK PEŁNEGO ZESTAWU KOLUMN Q/A – pomijam sekcję FAQ")
    print("    (potrzebne kolumny logiczne: question + answer)")
    print("=" * 60)

print("=" * 60)
print("=== BADANIE FAQ ZAKOŃCZONE ===")
print("=" * 60)

KOLUMNY W DF: ['\ufeffSklep / Marka', 'Oryginalny URL', 'Kategoria', 'Pytanie', 'Odpowiedź']
ZMAPOWANE KOLUMNY QA_COLS: {'shop': None, 'url': None, 'q': 'Pytanie', 'a': 'Odpowiedź'}
1. PODSTAWOWE INFORMACJE O TABELI (WSZYSTKIE REKORDY)
Liczba wierszy: 44
Liczba kolumn: 5
Całkowita liczba obserwacji: 220
2. NAZWY KOLUMN
['\ufeffSklep / Marka', 'Oryginalny URL', 'Kategoria', 'Pytanie', 'Odpowiedź']
3. PODGLĄD DANYCH (pierwsze 3 wiersze)


,﻿Sklep / Marka,Oryginalny URL,Kategoria,Pytanie,Odpowiedź
0,FORTE Meble,https://forte.com.pl,Dystrybucja i zakup,Gdzie mogę zakupić meble marki FORTE?,Meble FORTE sprzedawane są przez sieć partnerską w ponad 1000 salonów meblowych na terenie całej Polski. Pełną listę punktów sprzedaży można znaleźć na naszej stronie internetowej w zakładce 'Gdzi...
1,FORTE Meble,https://forte.com.pl,Bezpieczeństwo i montaż,Czy meble FORTE muszą być mocowane do ściany?,"Tak, dla bezpieczeństwa użytkowników (szczególnie dzieci) większość wysokich mebli, takich jak komody, regały czy szafy, posiada w zestawie specjalne okucia przeznaczone do montażu ściennego. Kate..."
2,Szynaka Meble,https://szynaka.pl/pytania-i-odpowiedzi/,Zakup i Dostępność,Gdzie można kupić meble?,Produkty firmy Szynaka Meble można kupić w ponad 350 salonach meblowych w całej Polsce. Sklep znajdujący się najbliżej Twojego miejsca zamieszkania znajdziesz w zakładce „Gdzie kupić?”.


4. PODGLĄD DANYCH (ostatnie 3 wiersze)


,﻿Sklep / Marka,Oryginalny URL,Kategoria,Pytanie,Odpowiedź
0,Dekoria,https://dekoria.pl,Darmowe próbki,Czy można zamówić próbki tkanin przed zakupem i ile one kosztują?,"Tak, na stronie dekoria.pl/probnik można wybrać i zamówić do 10 darmowych próbek interesujących Cię tkanin. Wysyłamy je bezpłatnie za pośrednictwem Poczty Polskiej (list zwykły), aby ułatwić dopas..."
1,Dekoria,https://dekoria.pl,Faktury i prawo do zwrotu,Czy zamawiając towar na firmę (faktura VAT) przysługuje mi prawo do zwrotu?,"Nie. Towar zakupiony w ramach prowadzonej działalności gospodarczej (z podaniem numeru NIP firmy) nie podlega standardowemu konsumenckiemu prawu do zwrotu bez podania przyczyny, ponieważ kupujący ..."
2,Fabryka Form,https://fabrykaform.pl,Oryginalność produktów,Czy wszystkie produkty oferowane w Fabryce Form są produktami oryginalnymi?,"Tak, jesteśmy oficjalnym dystrybutorem wszystkich marek premium prezentowanych w naszym sklepie (m.in. Alessi, Stelton, Joseph Joseph). Wszystkie towary są w 100% oryginalne, fabrycznie nowe i obj..."


5. TYPY DANYCH I BRAKUJĄCE WARTOŚCI (WSZYSTKIE REKORDY)


...

A4. BRAKI W PYTANIACH I ODPOWIEDZIACH (WSZYSTKIE REKORDY)
Brakujące (NaN) PYTANIA:    0
Brakujące (NaN) ODPOWIEDZI: 0

Przykładowe rekordy z brakami (max 3):


,﻿Sklep / Marka,Oryginalny URL,Kategoria,Pytanie,Odpowiedź,question_len,answer_len


A6. PIERWSZE SŁOWO PYTANIA (WSZYSTKIE REKORDY)
Najczęstsze pierwsze słowa w pytaniach:
q_first_word
czy         21
jak         10
gdzie        4
co           3
jakie        2
z            1
ile          1
dlaczego     1
jaki         1
Name: count, dtype: int64

Przykładowe pytania dla najpopularniejszego pierwszego słowa (max 3):


,Pytanie,Odpowiedź
0,Czy meble FORTE muszą być mocowane do ściany?,"Tak, dla bezpieczeństwa użytkowników (szczególnie dzieci) większość wysokich mebli, takich jak komody, regały czy szafy, posiada w zestawie specjalne okucia przeznaczone do montażu ściennego. Kate..."
1,"Czy meble dostępne w ofercie są zmontowane, czy sprzedawane w paczkach do samodzielnego montażu?","Firma posiada w swojej ofercie zarówno meble zmontowane, jak i bryły przeznaczone do samodzielnego montażu. Przy interesujących Cię bryłach znajdziesz piktogramy zawierające tę informację."
2,Czy można zamówić mebel w innych kolorach niż jest on przedstawiony na zdjęciu?,"W opisie produktu na stronie internetowej zawarta jest szczegółowa informacja, w jakich wybarwieniach kolorystycznych dostępny jest dany produkt."


=== BADANIE FAQ ZAKOŃCZONE ===


In [ ]:
# === KOMPLEKSOWE, ELASTYCZNE BADANIE FAQ / CZAT CSV + ZAPIS POD RAG ===

import pandas as pd
import re
import os

# 1. ŚCIEŻKA DO WEJŚCIOWEGO CSV (ZMIENIASZ TYLKO TO)
csv_path = r"C:\1\T4\jdszr24-grupa-4\SKRAPOWANIE\SKRAP_DED\DATA\faq_output.csv"

# 2. Wczytanie danych (obsługa polskich znaków, autodetekcja separatora)
df = pd.read_csv(
    csv_path,
    encoding="utf-8",   # jeśli będą krzaki, zmień na "cp1250"
    engine="python",
    sep=None,           # autodetekcja separatora
)

# 3. Ustawienia wyświetlania (czytelność)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 140)

print("KOLUMNY W DF:", list(df.columns))

# 4. Automatyczna konfiguracja kolumn FAQ / czatu (odporna na nazwy)
cols_lower = {c.lower().strip(): c for c in df.columns}

def find_col(candidates):
    for cand in candidates:
        if cand in cols_lower:
            return cols_lower[cand]
    return None

QA_COLS = {
    "shop": find_col(["shop", "sklep"]),
    "url":  find_col(["source_url", "url", "link", "href"]),
    "q":    find_col(["question", "pytanie", "query", "user_message"]),
    "a":    find_col(["answer", "odpowiedz", "odpowiedź", "response", "bot_message"]),
}

print("ZMAPOWANE KOLUMNY QA_COLS:", QA_COLS)

# 5. Funkcja czyszcząca tekst (HTML, encje, whitespace)
def clean_text(s: str) -> str:
    if pd.isna(s):
        return s
    s = str(s)

    # usuń proste znaczniki HTML
    s = re.sub(r"<[^>]+>", " ", s)

    # podstawowe encje HTML
    html_entities = {
        "&nbsp;": " ",
        "&amp;": "&",
        "&quot;": '"',
        "&apos;": "'",
        "&lt;": "<",
        "&gt;": ">",
    }
    for ent, rep in html_entities.items():
        s = s.replace(ent, rep)

    # twarde spacje, CR
    s = s.replace("\xa0", " ")
    s = s.replace("\r", " ")

    # wielokrotne białe znaki → jedna spacja
    s = re.sub(r"\s+", " ", s)

    # przycięcie
    return s.strip()

# 6. Czyszczenie tekstu we wszystkich kolumnach tekstowych (pełna elastyczność)
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].apply(clean_text)
    df.loc[df[col].astype(str).str.strip() == "", col] = pd.NA

# 7. Podstawowe informacje (na całym df)
print("=" * 60)
print("1. PODSTAWOWE INFORMACJE O TABELI (WSZYSTKIE REKORDY)")
print("=" * 60)
print(f"Liczba wierszy: {df.shape[0]}")
print(f"Liczba kolumn: {df.shape[1]}")
print(f"Całkowita liczba obserwacji: {df.shape[0] * df.shape[1]}")

# 8. Nazwy kolumn
print("=" * 60)
print("2. NAZWY KOLUMN")
print("=" * 60)
print(df.columns.tolist())

# 9. Podgląd: 3 pierwsze i 3 ostatnie wiersze
print("=" * 60)
print("3. PODGLĄD DANYCH (pierwsze 3 wiersze)")
print("=" * 60)
display(df.head(3).reset_index(drop=True))

print("=" * 60)
print("4. PODGLĄD DANYCH (ostatnie 3 wiersze)")
print("=" * 60)
display(df.tail(3).reset_index(drop=True))

# 10. Typy danych i braki – na całym df
print("=" * 60)
print("5. TYPY DANYCH I BRAKUJĄCE WARTOŚCI (WSZYSTKIE REKORDY)")
print("=" * 60)
info_df = pd.DataFrame({
    "Typ danych": df.dtypes,
    "Liczba brakujących": df.isnull().sum(),
    "% brakujących": (df.isnull().sum() / len(df) * 100).round(2),
    "Unikalne wartości": df.nunique()
})
display(info_df.reset_index().rename(columns={"index": "Kolumna"}))

# 11. Statystyki numeryczne – całe df (jeśli są)
print("=" * 60)
print("6. STATYSTYKI OPISOWE (kolumny numeryczne, WSZYSTKIE REKORDY)")
print("=" * 60)
num = df.select_dtypes(include=["int64", "float64"])
if not num.empty:
    display(num.describe())
else:
    print("Brak kolumn numerycznych.")

# 12. Statystyki tekstowe – całe df
print("=" * 60)
print("7. STATYSTYKI TEKSTOWE (WSZYSTKIE REKORDY)")
print("=" * 60)
display(df.describe(include=["object", "string"]))

# 13. Zakres danych – całe df
print("=" * 60)
print("8. ZAKRES DANYCH (WSZYSTKIE REKORDY)")
print("=" * 60)
for col in df.columns:
    if df[col].dtype in ["int64", "float64"]:
        print(f"{col}: {df[col].min()} -> {df[col].max()}")
    elif df[col].dtype == "object":
        print(f"{col}: {df[col].nunique()} unikalnych wartości")

# 14. Specjalne EDA dla FAQ / czatu – tylko jeśli mamy Q/A
has_q = QA_COLS["q"] is not None
has_a = QA_COLS["a"] is not None
has_shop = QA_COLS["shop"] is not None
has_url = QA_COLS["url"] is not None

if has_q and has_a:
    print("=" * 60)
    print("A1. INFO O PYTANIACH I ODPOWIEDZIACH (WSZYSTKIE REKORDY)")
    print("=" * 60)

    n_rows = len(df)
    n_shops = df[QA_COLS["shop"]].nunique() if has_shop else None
    n_urls  = df[QA_COLS["url"]].nunique() if has_url else None
    n_q     = df[QA_COLS["q"]].nunique()
    n_a     = df[QA_COLS["a"]].nunique()

    print(f"Liczba rekordów:              {n_rows}")
    if has_shop:
        print(f"Liczba unikalnych sklepów:    {n_shops}")
    if has_url:
        print(f"Liczba unikalnych URL-i:      {n_urls}")
    print(f"Liczba unikalnych PYTAŃ:      {n_q}")
    print(f"Liczba unikalnych ODPOWIEDZI: {n_a}")

    # duplikaty – globalnie
    print("=" * 60)
    print("A2. DUPLIKATY PYTAŃ I PAR Q&A (WSZYSTKIE REKORDY)")
    print("=" * 60)

    dup_q  = df.duplicated(subset=[QA_COLS["q"]]).sum()
    dup_qa = df.duplicated(subset=[QA_COLS["q"], QA_COLS["a"]]).sum()

    print(f"Liczba zduplikowanych PYTAŃ:   {dup_q}")
    print(f"Liczba zduplikowanych PAR Q&A: {dup_qa}")

    # długości – globalnie
    print("=" * 60)
    print("A3. DŁUGOŚCI PYTAŃ I ODPOWIEDZI (WSZYSTKIE REKORDY)")
    print("=" * 60)

    df["question_len"] = df[QA_COLS["q"]].astype(str).str.len()
    df["answer_len"]   = df[QA_COLS["a"]].astype(str).str.len()

    print("Statystyki długości PYTAŃ:")
    print(df["question_len"].describe())

    print("\nStatystyki długości ODPOWIEDZI:")
    print(df["answer_len"].describe())

    print("\nNajkrótsze PYTANIA (top 3):")
    display(
        df[[QA_COLS["q"], "question_len"]]
        .sort_values("question_len")
        .head(3)
        .reset_index(drop=True)
    )

    print("\nNajdłuższe PYTANIA (top 3):")
    display(
        df[[QA_COLS["q"], "question_len"]]
        .sort_values("question_len", ascending=False)
        .head(3)
        .reset_index(drop=True)
    )

    print("\nNajkrótsze ODPOWIEDZI (top 3):")
    display(
        df[[QA_COLS["a"], "answer_len"]]
        .sort_values("answer_len")
        .head(3)
        .reset_index(drop=True)
    )

    print("\nNajdłuższe ODPOWIEDZI (top 3):")
    display(
        df[[QA_COLS["a"], "answer_len"]]
        .sort_values("answer_len", ascending=False)
        .head(3)
        .reset_index(drop=True)
    )

    # braki – globalnie
    print("=" * 60)
    print("A4. BRAKI W PYTANIACH I ODPOWIEDZIACH (WSZYSTKIE REKORDY)")
    print("=" * 60)

    missing_q = df[QA_COLS["q"]].isna().sum()
    missing_a = df[QA_COLS["a"]].isna().sum()

    print(f"Brakujące (NaN) PYTANIA:    {missing_q}")
    print(f"Brakujące (NaN) ODPOWIEDZI: {missing_a}")

    bad_rows = df[df[QA_COLS["q"]].isna() | df[QA_COLS["a"]].isna()]
    print("\nPrzykładowe rekordy z brakami (max 3):")
    display(bad_rows.head(3).reset_index(drop=True))

else:
    print("=" * 60)
    print("A*. BRAK PEŁNEGO ZESTAWU KOLUMN Q/A – pomijam sekcję FAQ")
    print("    (potrzebne kolumny logiczne: question + answer)")
    print("=" * 60)

print("=" * 60)
print("=== BADANIE FAQ ZAKOŃCZONE ===")
print("=" * 60)

# 15. DYNAMICZNY ZAPIS OCZYSZCZONEGO PLIKU POD RAG / EMBEDDINGI

base_name = os.path.basename(csv_path)           # np. 'faq_output.csv'
name_no_ext, ext = os.path.splitext(base_name)   # 'faq_output', '.csv'
out_dir = os.path.dirname(csv_path)

out_filename_rag  = f"{name_no_ext}__clean_for_rag{ext}"
out_filename_full = f"{name_no_ext}__clean_full{ext}"

out_path_rag  = os.path.join(out_dir, out_filename_rag)
out_path_full = os.path.join(out_dir, out_filename_full)

cols_for_rag = []

if QA_COLS.get("q") is not None:
    cols_for_rag.append(QA_COLS["q"])
if QA_COLS.get("a") is not None:
    cols_for_rag.append(QA_COLS["a"])
if QA_COLS.get("url") is not None:
    cols_for_rag.append(QA_COLS["url"])
if QA_COLS.get("shop") is not None:
    cols_for_rag.append(QA_COLS["shop"])

if cols_for_rag:
    df_rag = df[cols_for_rag].copy()
    df_rag = df_rag.dropna(subset=[QA_COLS["q"], QA_COLS["a"]], how="any")
    df_rag = df_rag.reset_index(drop=True)
    df_rag.to_csv(out_path_rag, index=False, encoding="utf-8")
    print(f"Oczyszczony plik pod RAG zapisany do:\n{out_path_rag}")
    print(f"Liczba zapisanych par Q&A: {len(df_rag)}")
else:
    df.to_csv(out_path_full, index=False, encoding="utf-8")
    print("Nie znaleziono pełnego zestawu kolumn Q/A.")
    print(f"Zapisano pełny oczyszczony DataFrame do:\n{out_path_full}")
    

KOLUMNY W DF: ['\ufeffSklep / Marka', 'Oryginalny URL', 'Kategoria', 'Pytanie', 'Odpowiedź']
ZMAPOWANE KOLUMNY QA_COLS: {'shop': None, 'url': None, 'q': 'Pytanie', 'a': 'Odpowiedź'}
1. PODSTAWOWE INFORMACJE O TABELI (WSZYSTKIE REKORDY)
Liczba wierszy: 44
Liczba kolumn: 5
Całkowita liczba obserwacji: 220
2. NAZWY KOLUMN
['\ufeffSklep / Marka', 'Oryginalny URL', 'Kategoria', 'Pytanie', 'Odpowiedź']
3. PODGLĄD DANYCH (pierwsze 3 wiersze)


,﻿Sklep / Marka,Oryginalny URL,Kategoria,Pytanie,Odpowiedź
0,FORTE Meble,https://forte.com.pl,Dystrybucja i zakup,Gdzie mogę zakupić meble marki FORTE?,Meble FORTE sprzedawane są przez sieć partnerską w ponad 1000 salonów meblowych na terenie całej Polski. Pełną listę punktów sprzedaży można znaleźć na naszej stronie internetowej w zakładce 'Gdzi...
1,FORTE Meble,https://forte.com.pl,Bezpieczeństwo i montaż,Czy meble FORTE muszą być mocowane do ściany?,"Tak, dla bezpieczeństwa użytkowników (szczególnie dzieci) większość wysokich mebli, takich jak komody, regały czy szafy, posiada w zestawie specjalne okucia przeznaczone do montażu ściennego. Kate..."
2,Szynaka Meble,https://szynaka.pl/pytania-i-odpowiedzi/,Zakup i Dostępność,Gdzie można kupić meble?,Produkty firmy Szynaka Meble można kupić w ponad 350 salonach meblowych w całej Polsce. Sklep znajdujący się najbliżej Twojego miejsca zamieszkania znajdziesz w zakładce „Gdzie kupić?”.


4. PODGLĄD DANYCH (ostatnie 3 wiersze)


,﻿Sklep / Marka,Oryginalny URL,Kategoria,Pytanie,Odpowiedź
0,Dekoria,https://dekoria.pl,Darmowe próbki,Czy można zamówić próbki tkanin przed zakupem i ile one kosztują?,"Tak, na stronie dekoria.pl/probnik można wybrać i zamówić do 10 darmowych próbek interesujących Cię tkanin. Wysyłamy je bezpłatnie za pośrednictwem Poczty Polskiej (list zwykły), aby ułatwić dopas..."
1,Dekoria,https://dekoria.pl,Faktury i prawo do zwrotu,Czy zamawiając towar na firmę (faktura VAT) przysługuje mi prawo do zwrotu?,"Nie. Towar zakupiony w ramach prowadzonej działalności gospodarczej (z podaniem numeru NIP firmy) nie podlega standardowemu konsumenckiemu prawu do zwrotu bez podania przyczyny, ponieważ kupujący ..."
2,Fabryka Form,https://fabrykaform.pl,Oryginalność produktów,Czy wszystkie produkty oferowane w Fabryce Form są produktami oryginalnymi?,"Tak, jesteśmy oficjalnym dystrybutorem wszystkich marek premium prezentowanych w naszym sklepie (m.in. Alessi, Stelton, Joseph Joseph). Wszystkie towary są w 100% oryginalne, fabrycznie nowe i obj..."


5. TYPY DANYCH I BRAKUJĄCE WARTOŚCI (WSZYSTKIE REKORDY)


...


Najdłuższe ODPOWIEDZI (top 3):


,Odpowiedź,answer_len
0,"Reklamacji podlegają m.in. uszkodzenia transportowe, rysy, pęknięcia, nieszczelności urządzeń AGD lub niezgodność z opisem. Wymagany jest dowód zakupu (paragon/faktura lub potwierdzenie płatności ...",306
1,"Standardowo kurier ma obowiązek wniesienia przesyłki pod drzwi lokalu, jeżeli jej waga nie przekracza 30 kg. W przypadku paczek cięższych (np. duże szafy, łóżka, narożniki) kurier dostarcza przesy...",301
2,"Marka Kler oferuje bardzo szerokie możliwości personalizacji modułowej. Wiele kolekcji pozwala na dobór układu elementów, rodzaju i koloru skóry lub tkaniny premium, a także funkcji dodatkowych (n...",299


A4. BRAKI W PYTANIACH I ODPOWIEDZIACH (WSZYSTKIE REKORDY)
Brakujące (NaN) PYTANIA:    0
Brakujące (NaN) ODPOWIEDZI: 0

Przykładowe rekordy z brakami (max 3):


,﻿Sklep / Marka,Oryginalny URL,Kategoria,Pytanie,Odpowiedź,question_len,answer_len


=== BADANIE FAQ ZAKOŃCZONE ===
Oczyszczony plik pod RAG zapisany do:
C:\1\T4\jdszr24-grupa-4\SKRAPOWANIE\SKRAP_DED\DATA\faq_output__clean_for_rag.csv
Liczba zapisanych par Q&A: 44
